In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import  re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import joblib

In [30]:
df=pd.read_csv('P:\AI-Resume-Screener\data\dataset_cleaned.csv')

In [31]:
def cleanResume(resumeText):
    resumeText = re.sub('http\S+\s*', ' ', resumeText)  # remove URLs
    resumeText = re.sub('RT|cc', ' ', resumeText)  # remove RT and cc
    resumeText = re.sub('#\S+', '', resumeText)  # remove hashtags
    resumeText = re.sub('@\S+', '  ', resumeText)  # remove mentions
    resumeText = re.sub('[%s]' % re.escape("""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~"""), ' ', resumeText)  # remove punctuations
    resumeText = re.sub(r'[^\x00-\x7f]',r' ', resumeText) 
    resumeText = re.sub('\s+', ' ', resumeText)  # remove extra whitespace
    return resumeText
df=df.drop(columns=['Transcript'])

In [32]:
df['Resume_cleaned'] = df['Resume'].apply(lambda x:cleanResume(x))
df['Job_Description_cleaned'] = df['Job_Description'].apply(lambda x:cleanResume(x))
df

,Role,Resume,decision,Job_Description,jaccard_resume_jd,jaccard_resume_transcript,jaccard_jd_transcript,jd_resume_similarity,jd_transcript_similarity,resume_transcript_similarity,resume_length,jd_length,Transcript_length,keyword_overlap_ratio_resume_jd,keyword_overlap_ratio_transcript_jd,Resume_cleaned,Job_Description_cleaned
0,E-commerce Specialist,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...,0.017544,0.070381,0.014286,0.110196,0.045556,0.212312,253,11,318,0.272727,0.272727,professional resume jason jones jason jones co...,passionate team forefront machine learning com...
1,Game Developer,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...,0.050000,0.046025,0.022222,0.157439,0.095865,0.117589,40,11,331,0.181818,0.454545,professional resume ann marshall ann marshall ...,help build generation products game developer ...
2,Human Resources Specialist,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...,0.028090,0.121813,0.030172,0.165360,0.043850,0.332292,286,13,362,0.384615,0.538462,professional resume patrick lain patrick lain ...,need human resources specialist enhance team t...
3,E-commerce Specialist,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...,0.020408,0.134897,0.011719,0.090639,0.068059,0.252923,234,11,467,0.272727,0.272727,professional resume patricia gray patricia gra...,passionate team forefront cloud computing comm...
4,E-commerce Specialist,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...,0.018182,0.113565,0.019512,0.035783,0.055557,0.195964,271,12,323,0.250000,0.333333,professional resume amanda gross amanda gross ...,looking experienced commerce specialist join t...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10169,Product Manager,sample resume diana miller diana miller contac...,reject,comprehensive job description product manager ...,0.186335,0.106007,0.101852,0.430270,0.168836,0.276525,310,406,273,0.147783,0.081281,sample resume diana miller diana miller contac...,comprehensive job description product manager ...
10170,UI Engineer,sample resume grace taylor grace taylor contac...,reject,sample job description ui engineer job title u...,0.228571,0.087209,0.084011,0.435852,0.091417,0.199382,319,378,296,0.211640,0.082011,sample resume grace taylor grace taylor contac...,sample job description ui engineer job title u...
10171,UI Engineer,sample resume hank brown hank brown ui enginee...,select,job description ui engineer role job title ui ...,0.178977,0.138425,0.171569,0.399942,0.325333,0.331898,343,329,485,0.191489,0.212766,sample resume hank brown hank brown ui enginee...,job description ui engineer role job title ui ...
10172,Data Engineer,sample resume diana wilson diana wilson contac...,reject,comprehensive job description data engineer ro...,0.200000,0.113879,0.128289,0.396570,0.311673,0.315879,317,366,286,0.174863,0.106557,sample resume diana wilson diana wilson contac...,comprehensive job description data engineer ro...


In [33]:
vectorizer = CountVectorizer(stop_words='english')
analyzer = vectorizer.build_analyzer()
df['Resume'] = df['Resume_cleaned'].apply(lambda x: ' '.join(analyzer(x)))
df['Job_Description'] = df['Job_Description_cleaned'].apply(lambda x: ' '.join(analyzer(x)))
df.head()


,Role,Resume,decision,Job_Description,jaccard_resume_jd,jaccard_resume_transcript,jaccard_jd_transcript,jd_resume_similarity,jd_transcript_similarity,resume_transcript_similarity,resume_length,jd_length,Transcript_length,keyword_overlap_ratio_resume_jd,keyword_overlap_ratio_transcript_jd,Resume_cleaned,Job_Description_cleaned
0,E-commerce Specialist,professional resume jason jones jason jones co...,reject,passionate team forefront machine learning com...,0.017544,0.070381,0.014286,0.110196,0.045556,0.212312,253,11,318,0.272727,0.272727,professional resume jason jones jason jones co...,passionate team forefront machine learning com...
1,Game Developer,professional resume ann marshall ann marshall ...,select,help build generation products game developer ...,0.050000,0.046025,0.022222,0.157439,0.095865,0.117589,40,11,331,0.181818,0.454545,professional resume ann marshall ann marshall ...,help build generation products game developer ...
2,Human Resources Specialist,professional resume patrick lain patrick lain ...,reject,need human resources specialist enhance team t...,0.028090,0.121813,0.030172,0.165360,0.043850,0.332292,286,13,362,0.384615,0.538462,professional resume patrick lain patrick lain ...,need human resources specialist enhance team t...
3,E-commerce Specialist,professional resume patricia gray patricia gra...,select,passionate team forefront cloud computing comm...,0.020408,0.134897,0.011719,0.090639,0.068059,0.252923,234,11,467,0.272727,0.272727,professional resume patricia gray patricia gra...,passionate team forefront cloud computing comm...
4,E-commerce Specialist,professional resume amanda gross amanda gross ...,reject,looking experienced commerce specialist join t...,0.018182,0.113565,0.019512,0.035783,0.055557,0.195964,271,12,323,0.250000,0.333333,professional resume amanda gross amanda gross ...,looking experienced commerce specialist join t...


In [34]:
df=df.drop(['Resume_cleaned','Job_Description_cleaned'],axis=1)

In [35]:
word_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
     ngram_range=(1, 2))
combined = pd.concat([
    df["Resume"],
    df["Job_Description"]
])
word_vectorizer.fit(combined)
resume_vectorized=word_vectorizer.transform(df["Resume"])
job_description_vectorized=word_vectorizer.transform(df["Job_Description"])

In [36]:

jd_resume_similarity = []
for i in range(len(df)):
    score = cosine_similarity(
        resume_vectorized[i],
        job_description_vectorized[i]
    )[0][0]

    jd_resume_similarity.append(score)
df['jd_resume_similarity'] = jd_resume_similarity

In [37]:
df["resume_length"] = df["Resume"].apply(lambda x: len(x.split()))
df["jd_length"] = df["Job_Description"].apply(lambda x: len(x.split()))

In [38]:
keyword_overlap_ratio=[]
stop_words = set(ENGLISH_STOP_WORDS)
for i in range(len(df)):
    job_words=df['Job_Description'][i].split(" ")
    resume_words=df['Resume'][i].split(" ")
    resume_words = {
        w for w in resume_words
        if w not in stop_words
    }

    jd_words = {
        w for w in job_words
        if w not in stop_words
    }
    comman=resume_words.intersection(jd_words)
    keyword_overlap_ratio.append(len(comman)/len(job_words))

df['keyword_overlap_ratio_resume_jd'] = keyword_overlap_ratio

In [39]:
df.to_csv('P:\AI-Resume-Screener\data\dataset_cleaned_nontranscript.csv',index=False)

In [40]:
joblib.dump(word_vectorizer, r'P:\AI-Resume-Screener\models\tfidf_vectorizer.pkl')

['P:\\AI-Resume-Screener\\models\\tfidf_vectorizer.pkl']